# Sri Lanka Weather Analytics - Shortwave Radiation Analysis

This notebook calculates the percentage of days with shortwave_radiation_sum > 15 MJ/m² per month across all districts.

**Requirements:** 4.1

## 1. Setup and Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, StringType
)
from pyspark.sql.functions import (
    col, when, count, sum as spark_sum, month, year, to_date,
    round as spark_round
)
import os

## 2. Configuration

In [ ]:
# Radiation threshold in MJ/m²
RADIATION_THRESHOLD = 15.0

# Data paths
weather_path = "../dataset/weatherData.csv"

print(f"Radiation threshold: {RADIATION_THRESHOLD} MJ/m²")

## 3. Create Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("ShortwaveRadiationAnalysis") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.session.timeZone", "Asia/Colombo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 4. Define Schema and Load Data

In [ ]:
# Weather data schema
weather_schema = StructType([
    StructField("location_id", IntegerType(), nullable=False),
    StructField("date", StringType(), nullable=False),
    StructField("weather_code", IntegerType(), nullable=True),
    StructField("temperature_2m_max", FloatType(), nullable=True),
    StructField("temperature_2m_min", FloatType(), nullable=True),
    StructField("temperature_2m_mean", FloatType(), nullable=True),
    StructField("apparent_temperature_max", FloatType(), nullable=True),
    StructField("apparent_temperature_min", FloatType(), nullable=True),
    StructField("apparent_temperature_mean", FloatType(), nullable=True),
    StructField("daylight_duration", FloatType(), nullable=True),
    StructField("sunshine_duration", FloatType(), nullable=True),
    StructField("precipitation_sum", FloatType(), nullable=True),
    StructField("rain_sum", FloatType(), nullable=True),
    StructField("precipitation_hours", FloatType(), nullable=True),
    StructField("wind_speed_10m_max", FloatType(), nullable=True),
    StructField("wind_gusts_10m_max", FloatType(), nullable=True),
    StructField("wind_direction_10m_dominant", FloatType(), nullable=True),
    StructField("shortwave_radiation_sum", FloatType(), nullable=True),
    StructField("et0_fao_evapotranspiration", FloatType(), nullable=True),
    StructField("sunrise", StringType(), nullable=True),
    StructField("sunset", StringType(), nullable=True)
])

# Load weather data
weather_df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "") \
    .option("nanValue", "NaN") \
    .schema(weather_schema) \
    .csv(weather_path)

print(f"Weather records loaded: {weather_df.count()}")

## 5. Parse Dates and Extract Month

In [ ]:
# Parse date from M/D/YYYY format and extract month
weather_df = weather_df.withColumn(
    "parsed_date",
    to_date(col("date"), "M/d/yyyy")
)

weather_df = weather_df \
    .withColumn("year", year(col("parsed_date"))) \
    .withColumn("month", month(col("parsed_date")))

# Verify date parsing
invalid_dates = weather_df.filter(col("parsed_date").isNull()).count()
print(f"Records with invalid dates: {invalid_dates}")

## 6. Explore Shortwave Radiation Data

In [ ]:
# Check radiation data statistics
print("Shortwave radiation statistics:")
weather_df.select("shortwave_radiation_sum").describe().show()

# Check for null values
null_radiation = weather_df.filter(col("shortwave_radiation_sum").isNull()).count()
print(f"Records with null radiation values: {null_radiation}")

## 7. Calculate Radiation Percentage by Month

**Requirement 4.1:** Calculate the percentage of days where shortwave_radiation_sum exceeds 15 MJ/m² per month across all districts.

In [ ]:
def calculate_radiation_percentage_by_month(weather_df, threshold=RADIATION_THRESHOLD):
    """
    Calculate the percentage of days with shortwave_radiation_sum > threshold per month.
    
    This function groups data by month across all districts and calculates:
    - Total number of days with valid radiation readings
    - Number of days exceeding the threshold
    - Percentage of days exceeding the threshold
    
    Args:
        weather_df: DataFrame with weather data including shortwave_radiation_sum
        threshold: Radiation threshold in MJ/m² (default: 15.0)
        
    Returns:
        DataFrame with columns: month, total_days, days_above_threshold, percentage
        
    Validates: Requirements 4.1
    """
    # Filter out records with null radiation values
    valid_radiation_df = weather_df.filter(col("shortwave_radiation_sum").isNotNull())
    
    # Calculate days above threshold indicator
    radiation_analysis = valid_radiation_df.withColumn(
        "above_threshold",
        when(col("shortwave_radiation_sum") > threshold, 1).otherwise(0)
    )
    
    # Group by month and calculate statistics
    monthly_stats = radiation_analysis.groupBy("month").agg(
        count("*").alias("total_days"),
        spark_sum("above_threshold").alias("days_above_threshold")
    )
    
    # Calculate percentage
    result = monthly_stats.withColumn(
        "percentage",
        spark_round(
            (col("days_above_threshold") / col("total_days")) * 100,
            2
        )
    )
    
    # Order by month
    return result.orderBy("month")

print("Function defined successfully.")

In [ ]:
# Calculate radiation percentages
radiation_results = calculate_radiation_percentage_by_month(weather_df)

print(f"\nPercentage of days with shortwave radiation > {RADIATION_THRESHOLD} MJ/m² by month:")
radiation_results.show()

## 8. Format Results with Month Names

In [ ]:
# Month names for display
month_names = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

# Collect results and format
results = radiation_results.collect()

print("\n" + "=" * 70)
print(f"Shortwave Radiation Analysis - Days with radiation > {RADIATION_THRESHOLD} MJ/m²")
print("=" * 70)
print(f"{'Month':<12} {'Total Days':>12} {'Days Above':>15} {'Percentage':>12}")
print("-" * 70)

for row in results:
    month_name = month_names[row["month"] - 1]
    print(f"{month_name:<12} {row['total_days']:>12} {row['days_above_threshold']:>15} {row['percentage']:>11.2f}%")

print("=" * 70)

## 9. Summary Statistics

In [ ]:
# Calculate overall statistics
total_days = sum(row["total_days"] for row in results)
total_above = sum(row["days_above_threshold"] for row in results)
overall_percentage = (total_above / total_days) * 100 if total_days > 0 else 0

print(f"\nOverall Statistics:")
print(f"  Total days analyzed: {total_days}")
print(f"  Days with radiation > {RADIATION_THRESHOLD} MJ/m²: {total_above}")
print(f"  Overall percentage: {overall_percentage:.2f}%")

# Find months with highest and lowest percentages
max_month = max(results, key=lambda x: x["percentage"])
min_month = min(results, key=lambda x: x["percentage"])

print(f"\n  Highest radiation month: {month_names[max_month['month'] - 1]} ({max_month['percentage']:.2f}%)")
print(f"  Lowest radiation month: {month_names[min_month['month'] - 1]} ({min_month['percentage']:.2f}%)")

## 10. Cleanup

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped. Analysis complete!")

---

## Analysis Complete!

This notebook calculated the percentage of days with shortwave radiation exceeding 15 MJ/m² for each month across all districts in Sri Lanka.

**Key Findings:**
- The analysis covers all 12 months
- Results show seasonal variation in solar radiation intensity
- Higher percentages indicate months with more intense solar radiation

**Validates: Requirements 4.1**